# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Roselyn-Koech/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule

I will prioritize content for refresh review when it shows weaker search performance or visibility in the decision window. The baseline score combines CTR, average search position, and 90 day search impressions. Higher scores indicate a stronger reason to review the content for refresh.

Reason codes

LOW_CTR — the content receives relatively few clicks compared with its impressions.
WEAK_POSITION — the content has a relatively poor average search position.
LOW_VISIBILITY — the content has relatively low search impressions.

Action

Higher scoring content will be labelled REFRESH_REVIEW, while lower scoring content will be labelled MONITOR.

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

In [18]:
from google.colab import files

uploaded = files.upload()


Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv


In [23]:
DATA_PATH = Path("/content/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully.
Rows: 30000
Columns: 44


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [26]:
required_columns = [
    "ctr",
    "avg_position",
    "impressions_90d"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns found:")
for col in required_columns:
    print("✓", col)




Required columns found:
✓ ctr
✓ avg_position
✓ impressions_90d


In [27]:
# Show possible columns related to CTR, position, and impressions

for col in df.columns:
    col_lower = col.lower()

    if (
        "ctr" in col_lower
        or "click" in col_lower
        or "position" in col_lower
        or "impression" in col_lower
        or "visibility" in col_lower
    ):
        print(col)

impressions_90d
clicks_90d
days_with_impressions
impressions_last_30d
clicks_last_30d
impressions_prev_30d
clicks_prev_30d
ctr
avg_position
impression_tier
position_tier


In [28]:
df["ctr"] = pd.to_numeric(
    df["ctr"],
    errors="coerce"
)

df["avg_position"] = pd.to_numeric(
    df["avg_position"],
    errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
)


In [29]:
df["LOW_CTR"] = (
    1 - df["ctr"].rank(pct=True)
)

df["WEAK_POSITION"] = (
    df["avg_position"].rank(pct=True)
)

df["LOW_VISIBILITY"] = (
    1 - df["impressions_90d"].rank(pct=True)
)

In [30]:
signal_columns = [
    "LOW_CTR",
    "WEAK_POSITION",
    "LOW_VISIBILITY"
]

df[signal_columns] = (
    df[signal_columns].fillna(0)
)

In [32]:
# The baseline weights are:
# CTR              = 40%
# Average position = 35%
# Visibility       = 25%
# Higher score = stronger reason for refresh review.


df["baseline_action_score"] = (
    0.40 * df["LOW_CTR"]
    + 0.35 * df["WEAK_POSITION"]
    + 0.25 * df["LOW_VISIBILITY"]
)


In [33]:
df["rank"] = (
    df["baseline_action_score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

In [34]:
def get_reason_code(row):

    reasons = {
        "LOW_CTR": row["LOW_CTR"],
        "WEAK_POSITION": row["WEAK_POSITION"],
        "LOW_VISIBILITY": row["LOW_VISIBILITY"]
    }

    strongest_reason = max(
        reasons,
        key=reasons.get
    )

    return strongest_reason


df["reason_code"] = df.apply(
    get_reason_code,
    axis=1
)

In [35]:
review_cutoff = max(
    1,
    int(np.ceil(len(df) * 0.20))
)

df["action"] = np.where(
    df["rank"] <= review_cutoff,
    "REFRESH_REVIEW",
    "MONITOR"
)


In [36]:
df = df.sort_values(
    by="rank",
    ascending=True
).reset_index(drop=True)



In [37]:
OUTPUT_DIR = Path(
    "/content/work/outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [38]:
OUTPUT_PATH = (
    OUTPUT_DIR /
    "baseline_action_score.csv"
)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

In [39]:

print("\nBaseline scoring complete.")
print("--------------------------------")
print("Total content:", len(df))
print(
    "REFRESH_REVIEW:",
    (df["action"] == "REFRESH_REVIEW").sum()
)
print(
    "MONITOR:",
    (df["action"] == "MONITOR").sum()
)
print("Output:", OUTPUT_PATH)


Baseline scoring complete.
--------------------------------
Total content: 30000
REFRESH_REVIEW: 6000
MONITOR: 24000
Output: /content/work/outputs/baseline_action_score.csv


In [40]:
print("\nTop 20 content for review:")

display(
    df[
        [
            "rank",
            "ctr",
            "avg_position",
            "impressions_90d",
            "baseline_action_score",
            "action",
            "reason_code"
        ]
    ].head(20)
)



Top 20 content for review:


,rank,ctr,avg_position,impressions_90d,baseline_action_score,action,reason_code
0,1,0.0,245.0,1,0.907430,REFRESH_REVIEW,WEAK_POSITION
1,2,0.0,184.0,1,0.907418,REFRESH_REVIEW,WEAK_POSITION
2,3,0.0,161.0,1,0.907395,REFRESH_REVIEW,WEAK_POSITION
3,4,0.0,142.0,1,0.907360,REFRESH_REVIEW,WEAK_POSITION
4,5,0.0,110.0,1,0.907307,REFRESH_REVIEW,WEAK_POSITION
5,6,0.0,99.0,1,0.907255,REFRESH_REVIEW,WEAK_POSITION
6,7,0.0,98.0,1,0.907208,REFRESH_REVIEW,WEAK_POSITION
7,8,0.0,98.0,1,0.907208,REFRESH_REVIEW,WEAK_POSITION
8,9,0.0,97.0,1,0.907185,REFRESH_REVIEW,WEAK_POSITION
9,10,0.0,94.0,1,0.907092,REFRESH_REVIEW,WEAK_POSITION


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [41]:
top20 = df.sort_values(
    by="baseline_action_score",
    ascending=False
).head(20).copy()


In [42]:
def confidence_note(row):
    score = row["baseline_action_score"]

    if score >= 0.85:
        return "Higher baseline score; directionally stronger review candidate."
    elif score >= 0.70:
        return "Moderate baseline score; useful for review but needs validation."
    else:
        return "Lower baseline score; treat as a monitoring candidate."

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)


In [43]:
def wrong_if(row):
    reason = row["reason_code"]

    if reason == "LOW_CTR":
        return (
            "The zero or low CTR may reflect very low impressions "
            "rather than a genuine content problem."
        )

    elif reason == "WEAK_POSITION":
        return (
            "The position signal may not indicate a real content issue "
            "if the page has very little search activity."
        )

    elif reason == "LOW_VISIBILITY":
        return (
            "Low impressions may be expected for the page rather than "
            "evidence that the content needs refreshing."
        )

    return (
        "The recommendation could be wrong if the search signals "
        "do not represent a meaningful content opportunity."
    )


top20["what_would_make_it_wrong"] = top20.apply(
    wrong_if,
    axis=1
)


In [44]:
top20_review = top20[
    [
        "rank",
        "ctr",
        "avg_position",
        "impressions_90d",
        "baseline_action_score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,rank,ctr,avg_position,impressions_90d,baseline_action_score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,0.0,245.0,1,0.907430,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
1,2,0.0,184.0,1,0.907418,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
2,3,0.0,161.0,1,0.907395,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
3,4,0.0,142.0,1,0.907360,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
4,5,0.0,110.0,1,0.907307,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
5,6,0.0,99.0,1,0.907255,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
7,8,0.0,98.0,1,0.907208,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
6,7,0.0,98.0,1,0.907208,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
8,9,0.0,97.0,1,0.907185,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
9,10,0.0,94.0,1,0.907092,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...


In [54]:
top20["what_would_make_it_wrong"] = top20.apply(
    wrong_if,
    axis=1
)

In [55]:
# Display the completed Top-20 review

top20_review = top20[
    [
        "rank",
        "ctr",
        "avg_position",
        "impressions_90d",
        "baseline_action_score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,rank,ctr,avg_position,impressions_90d,baseline_action_score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,0.0,245.0,1,0.907430,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
1,2,0.0,184.0,1,0.907418,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
2,3,0.0,161.0,1,0.907395,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
3,4,0.0,142.0,1,0.907360,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
4,5,0.0,110.0,1,0.907307,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
5,6,0.0,99.0,1,0.907255,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
7,8,0.0,98.0,1,0.907208,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
6,7,0.0,98.0,1,0.907208,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
8,9,0.0,97.0,1,0.907185,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...
9,10,0.0,94.0,1,0.907092,REFRESH_REVIEW,WEAK_POSITION,Higher baseline score; directionally stronger ...,The position signal may not indicate a real co...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [45]:
weak_picks = top20[
    (
        (top20["ctr"] == 0) &
        (top20["impressions_90d"] <= top20["impressions_90d"].median())
    )
].copy()

print("Potentially weak picks:", len(weak_picks))

display(
    weak_picks[
        [
            "rank",
            "ctr",
            "avg_position",
            "impressions_90d",
            "baseline_action_score",
            "action",
            "reason_code"
        ]
    ]
)


Potentially weak picks: 20


,rank,ctr,avg_position,impressions_90d,baseline_action_score,action,reason_code
0,1,0.0,245.0,1,0.907430,REFRESH_REVIEW,WEAK_POSITION
1,2,0.0,184.0,1,0.907418,REFRESH_REVIEW,WEAK_POSITION
2,3,0.0,161.0,1,0.907395,REFRESH_REVIEW,WEAK_POSITION
3,4,0.0,142.0,1,0.907360,REFRESH_REVIEW,WEAK_POSITION
4,5,0.0,110.0,1,0.907307,REFRESH_REVIEW,WEAK_POSITION
5,6,0.0,99.0,1,0.907255,REFRESH_REVIEW,WEAK_POSITION
7,8,0.0,98.0,1,0.907208,REFRESH_REVIEW,WEAK_POSITION
6,7,0.0,98.0,1,0.907208,REFRESH_REVIEW,WEAK_POSITION
8,9,0.0,97.0,1,0.907185,REFRESH_REVIEW,WEAK_POSITION
9,10,0.0,94.0,1,0.907092,REFRESH_REVIEW,WEAK_POSITION


In [47]:
print("\nWeak pick interpretation:")
print(
    "Several high-ranked items have zero CTR and very low "
    "90-day impressions. These may be directional refresh "
    "candidates, but the low search volume means the baseline "
    "may not provide enough evidence that the content itself "
    "needs refreshing.")


Weak pick interpretation:
Several high-ranked items have zero CTR and very low 90-day impressions. These may be directional refresh candidates, but the low search volume means the baseline may not provide enough evidence that the content itself needs refreshing.


In [48]:
# The baseline should use only the decision-window signals:
#   ctr
#   avg_position
#   impressions_90d
#
# We should not use later performance outcomes such as
# last_30d or future-period columns to calculate the score.

The baseline should use only the decision-window signals:
ctr
avg_position
 impressions_90d
 We should not use later performance outcomes such as
last_30d or future-period columns to calculate the score.

In [49]:
score_columns = [
    "ctr",
    "avg_position",
    "impressions_90d"
]

print("\nColumns used to calculate the baseline score:")
for col in score_columns:
    print("✓", col)



Columns used to calculate the baseline score:
✓ ctr
✓ avg_position
✓ impressions_90d


In [50]:
future_or_comparison_columns = [
    col for col in df.columns
    if (
        "last_30d" in col.lower()
        or "prev_30d" in col.lower()
        or "future" in col.lower()
        or "next" in col.lower()
    )
]

print("\nPotential future/comparison columns found in dataset:")
print(future_or_comparison_columns)

used_future_columns = [
    col for col in future_or_comparison_columns
    if col in score_columns
]

print("\nFuture/comparison columns used in score:")
print(used_future_columns)


Potential future/comparison columns found in dataset:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Future/comparison columns used in score:
[]


In [51]:
product_flag_columns = [
    col for col in df.columns
    if "product" in col.lower() and "flag" in col.lower()
]

print("\nProduct flag columns found:")
print(product_flag_columns)

used_product_flags = [
    col for col in product_flag_columns
    if col in score_columns
]

print("\nProduct flag columns used in score:")
print(used_product_flags)


Product flag columns found:
[]

Product flag columns used in score:
[]


In [52]:
if len(used_future_columns) == 0:
    print("\n✓ No future/comparison columns were used in the baseline score.")
else:
    print(
        "\nWARNING: A future/comparison column appears in the "
        "baseline score."
    )

if len(used_product_flags) == 0:
    print("✓ No product flags were used in the baseline score.")
else:
    print(
        "WARNING: A product flag appears in the baseline score."
    )




✓ No future/comparison columns were used in the baseline score.
✓ No product flags were used in the baseline score.


In [53]:
print("\nSection 4 conclusion:")
print(
    "The baseline produces directional refresh candidates. "
    "The weakest picks are those with very low search activity, "
    "because low impressions can make the signals less reliable. "
    "The score itself uses CTR, average position, and 90-day "
    "impressions only, so later comparison windows and product "
    "flags are not used to calculate the baseline."
)


Section 4 conclusion:
The baseline produces directional refresh candidates. The weakest picks are those with very low search activity, because low impressions can make the signals less reliable. The score itself uses CTR, average position, and 90-day impressions only, so later comparison windows and product flags are not used to calculate the baseline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.